In [1]:
import pandas as pd
import re
import spacy
from tqdm import tqdm
from spacy.util import filter_spans
from spacy.language import Language

In [2]:
tqdm.pandas()

In [3]:
df_all = pd.read_csv("../data/processed/all_clean_for_ner.csv")

In [4]:
df_all.head()

,source,speaker,text
0,talkmap,agent,"You're welcome, Mistie. I apologize again for ..."
1,talkmap,client,That sounds reassuring. But what if someone ha...
2,talkmap,client,"Alright, thank you for your help, Dayna. I app..."
3,talkmap,agent,"Goodbye, Angeline. Have a great day."
4,talkmap,agent,"You're welcome, Lessie. Thank you for choosing..."


In [5]:
nlp = spacy.load("en_core_web_lg", disable=["tagger", "parser", "lemmatizer"])

In [6]:
ACCOUNT_RE = re.compile(r"\b[A-Z]{2,5}-?\d{4,}\b")
PHONE_RE = re.compile(r"\b(\+?\d{1,3}[-.\s]?)?\(?\d{2,4}\)?[-.\s]?\d{3,4}[-.\s]?\d{3,4}\b")
ERROR_RE = re.compile(r"\b[A-Z]{2,5}-?\d{2,}\b")


In [7]:
@Language.component("regex_entities")
def add_regex_entities(doc):
    ents = list(doc.ents)

    for match in ACCOUNT_RE.finditer(doc.text):
        span = doc.char_span(match.start(), match.end(), label="ACCOUNT_ID")
        if span:
            ents.append(span)

    for match in PHONE_RE.finditer(doc.text):
        span = doc.char_span(match.start(), match.end(), label="PHONE_NUMBER")
        if span:
            ents.append(span)

    doc.ents = filter_spans(ents)
    return doc


if "regex_entities" in nlp.pipe_names:
    nlp.remove_pipe("regex_entities")

nlp.add_pipe("regex_entities", last=True)


<function __main__.add_regex_entities(doc)>

In [8]:
if "regex_entities" in nlp.pipe_names:
    nlp.remove_pipe("regex_entities")
nlp.add_pipe("regex_entities", last=True)

<function __main__.add_regex_entities(doc)>

In [9]:
def make_relation(
    doc_id, subj, subj_type, rel,
    obj, obj_type, evidence,
    source, method, confidence
):
    return {
        "doc_id": doc_id,
        "subject": subj,
        "subject_type": subj_type,
        "relation": rel,
        "object": obj,
        "object_type": obj_type,
        "evidence_text": evidence,
        "confidence": confidence,
        "extraction_method": method,
        "source": source
    }


In [10]:
ISSUE_VERBS = {"have", "experience", "get", "lose", "drop", "face", "report"}
ACTIONS = {"reset", "replace", "refund", "cancel", "upgrade", "escalate"}
SERVICES = {"internet", "data plan", "broadband", "roaming", "voicemail", "subscription"}

In [ ]:
ISSUE_VERBS = {"have", "experience", "get", "lose", "drop", "face", "report"}

def extract_reported_issue(doc, row):
    if row["speaker"] != "client":
        return []

    rels = []
    for token in doc:
        if token.lemma_.lower() in ISSUE_VERBS:
            subj = [w for w in token.lefts if w.dep_ in {"nsubj", "nsubjpass"}]
            obj = [w for w in token.rights if w.dep_ in {"dobj", "attr", "pobj"}]

            if subj and obj:
                rels.append(make_relation(
                    row["doc_id"],
                    "CUSTOMER", "CUSTOMER",
                    "reported_issue",
                    obj[0].text, "ISSUE",
                    doc.text,
                    row["source"],
                    "dependency",
                    0.95   
                ))
    return rels



In [12]:
SERVICES = {"internet", "data", "broadband", "roaming", "voicemail", "subscription"}

def extract_uses_service(doc, row):
    rels = []
    for token in doc:
        if token.text.lower() in SERVICES:
            rels.append(make_relation(
                row["doc_id"], "CUSTOMER", "CUSTOMER",
                "uses_service", token.text.lower(), "SERVICE",
                doc.text, row["source"], "keyword", 0.80
            ))
    return rels


In [13]:
def extract_has_device(doc, row):
    rels = []
    for ent in doc.ents:
        if ent.label_ == "PRODUCT":
            rels.append(make_relation(
                row["doc_id"], "CUSTOMER", "CUSTOMER",
                "has_device", ent.text, "PRODUCT",
                doc.text, row["source"], "ner", 0.85
            ))
    return rels


In [14]:
def extract_account_relations(text, row):
    rels = []

    for acc in ACCOUNT_RE.findall(text):
        rels.append(make_relation(
            row["doc_id"], "ISSUE", "ISSUE",
            "related_to_account", acc, "ACCOUNT_ID",
            text, row["source"], "regex", 0.95
        ))

    for phone in PHONE_RE.findall(text):
        rels.append(make_relation(
            row["doc_id"], "CUSTOMER", "CUSTOMER",
            "related_to_account", phone, "PHONE_NUMBER",
            text, row["source"], "regex", 0.95
        ))

    return rels


In [15]:
def extract_temporal(doc, row):
    rels = []
    for ent in doc.ents:
        if ent.label_ in {"DATE", "TIME"}:
            rels.append(make_relation(
                row["doc_id"], "ISSUE", "ISSUE",
                "occurred_on", ent.text, ent.label_,
                doc.text, row["source"], "ner", 0.90
            ))
    return rels


In [16]:
ACTIONS = {"reset", "replace", "refund", "cancel", "upgrade", "escalate"}

def extract_actions(doc, row):
    if row["speaker"] != "agent":
        return []

    rels = []
    for token in doc:
        if token.lemma_.lower() in ACTIONS:
            rels.append(make_relation(
                row["doc_id"],
                "ISSUE", "ISSUE",
                "resolved_by",
                token.lemma_, "ACTION",
                doc.text,
                row["source"],
                "verb",
                0.90
            ))
    return rels



In [18]:
CHANNELS = {"call", "chat", "email", "ticket", "sms"}

def extract_misc(doc, row):
    rels = []

    for ent in doc.ents:
        if ent.label_ == "ORG":
            rels.append(make_relation(
                row["doc_id"], "TICKET", "TICKET",
                "escalated_to", ent.text, "TEAM",
                doc.text, row["source"], "ner", 0.75
            ))

        if ent.label_ in {"GPE", "LOC"}:
            rels.append(make_relation(
                row["doc_id"], "ISSUE", "ISSUE",
                "located_at", ent.text, "LOCATION",
                doc.text, row["source"], "ner", 0.85
            ))

        if ent.label_ == "MONEY":
            rels.append(make_relation(
                row["doc_id"], "ACCOUNT", "ACCOUNT",
                "has_billing_amount", ent.text, "MONEY",
                doc.text, row["source"], "ner", 0.85
            ))

    for match in ERROR_RE.finditer(doc.text):
        rels.append(make_relation(
            row["doc_id"], "ISSUE", "ISSUE",
            "has_error_code", match.group(), "ERROR_CODE",
            doc.text, row["source"], "regex", 0.95
        ))

    for token in doc:
        if token.text.lower() in CHANNELS:
            rels.append(make_relation(
                row["doc_id"], "CUSTOMER", "CUSTOMER",
                "reported_via", token.text.lower(), "CHANNEL",
                doc.text, row["source"], "keyword", 0.85
            ))

    return rels


In [19]:
all_relations = []

for i, row in tqdm(enumerate(df_all.to_dict(orient="records")), total=len(df_all)):
    row["doc_id"] = i
    doc = nlp(row["text"])

    all_relations.extend(extract_reported_issue(doc, row))
    all_relations.extend(extract_has_device(doc, row))
    all_relations.extend(extract_uses_service(doc, row))
    all_relations.extend(extract_account_relations(row["text"], row))
    all_relations.extend(extract_temporal(doc, row))
    all_relations.extend(extract_actions(doc, row))
    all_relations.extend(extract_misc(doc, row))


100%|██████████| 154824/154824 [40:42<00:00, 63.39it/s] 


In [24]:
df_relations = pd.DataFrame(all_relations)

In [23]:
print(df_relations["relation"].value_counts())
df_relations.sample(10)[["subject","relation","object","confidence","extraction_method"]]

relation
escalated_to          29351
occurred_on           22953
reported_via          17472
uses_service          16595
located_at             5261
has_billing_amount     2315
related_to_account     2001
has_device             1916
has_error_code          177
Name: count, dtype: int64


,subject,relation,object,confidence,extraction_method
11630,ISSUE,located_at,Antonio,0.85,ner
24095,TICKET,escalated_to,HDMI,0.75,ner
51299,TICKET,escalated_to,Kanisha,0.75,ner
38974,CUSTOMER,uses_service,data,0.80,keyword
49703,TICKET,escalated_to,Union Mobile,0.75,ner
15942,ISSUE,occurred_on,today,0.90,ner
16739,CUSTOMER,uses_service,data,0.80,keyword
70561,ISSUE,occurred_on,today,0.90,ner
76383,CUSTOMER,reported_via,chat,0.85,keyword
91300,CUSTOMER,uses_service,subscription,0.80,keyword


In [22]:
df_relations.to_csv("../data/processed/relations_extraction.csv", index=False)